In [1]:
import os
import numpy as np
import arcpy

def kalman_filter_2d(points, initial_state, initial_uncertainty, process_noise, measurement_noise):
    smoothed_points = []
    state_estimate = initial_state
    uncertainty_estimate = initial_uncertainty

    for point in points:
        z = point
        measurement = np.array([[z]])

        # Prediction step
        state_estimate = state_estimate  # No control input
        uncertainty_estimate = uncertainty_estimate + process_noise

        # Measurement update step
        kalman_gain = uncertainty_estimate @ np.linalg.inv(uncertainty_estimate + measurement_noise)
        state_estimate = state_estimate + kalman_gain @ (measurement - state_estimate)
        uncertainty_estimate = (np.eye(1) - kalman_gain) @ uncertainty_estimate

        smoothed_points.append(state_estimate.flatten())

    return np.array(smoothed_points)

# Parameters
initial_state = np.array([[0]])
initial_uncertainty = np.eye(1)
process_noise = np.eye(1) * 0.1
measurement_noise = np.eye(1) * 0.5

# Update these paths with your actual data paths
input_fc = r'C:\Users\purni\Documents\ArcGIS\Projects\For Phase 2_DEM_CS\Shape files\Waylogdata_MERGED_RAW_XYZ.shp'  # Path to your input shapefile
#output_fc = r'C:\Users\purni\Documents\Test\FilterPointsnew10.shp'  # Path to your output shapefile
fields = ['SHAPE@XY', 'Elevation']  # Ensure this matches the field names in your data


# Initialize a list to hold the points
points = []

# Read data from the input feature class
with arcpy.da.SearchCursor(input_fc, fields) as cursor:
    for row in cursor:
        #x, y = row[0]
        z = row[1]
        points.append([z])  # Append the point to the list        

# Convert the list of points to a NumPy array
#points = np.array(points)

# Apply the Kalman filter
smoothed_z = kalman_filter_2d(points, initial_state, initial_uncertainty, process_noise, measurement_noise)


#print(smoothed_z)


In [2]:
#Extract original XY values and write them to an array

xy_array = []

with arcpy.da.SearchCursor(input_fc, ["SHAPE@XY"]) as cursor:
    for row in cursor:
        xy_array.append(row[0])

#print(xy_array)

#stick the two arrays together
# Create the new list of (X, Y, Z) tuples
xyz_array = []

#Create the output XYZ array

for i in range(len(xy_array)):
    x, y = xy_array[i]
    z = smoothed_z[i].item()  # Extract the scalar value from the numpy array
    xyz_array.append((x, y, z))

smoothed_points = arcpy.Array()
smoothed_points = xyz_array

In [5]:
#Create a new featureclass to hold the smoothed data
# Set the workspace (geodatabase or folder where the feature class will be created)
arcpy.env.workspace = r"C:\Users\purni\Documents\ArcGIS\Projects\For Phase 2_DEM_CS\Shape files"

# Set environment to overwrite existing outputs
arcpy.env.overwriteOutput = True

# Specify the feature class name
Output_FC = "WL_Kalmanfilter_XYZ_POINTS"

# Specify the geometry type (e.g., Point, Multipoint, Polygon, Polyline)
geometry_type = "POINT"

#specify spatial Reference
spatial_reference = arcpy.SpatialReference("WGS 1984")

#template = '#'  # Use an existing feature class as a template if needed

# Create the feature class
arcpy.CreateFeatureclass_management(arcpy.env.workspace, Output_FC, geometry_type, spatial_reference=spatial_reference)


# Specify the existing feature class name
#existing_feature_class = "ExistingFeatureClass"

# Specify the new field name and data type
new_field_name = "Elevation"
data_type = "FLOAT"  # You can choose other data types like "LONG", "DOUBLE", etc.

# Add the new field to the existing feature class
arcpy.AddField_management(Output_FC, new_field_name, data_type)

# Insert smoothed points into the output feature class
with arcpy.da.InsertCursor(Output_FC, ['SHAPE@XY', 'Elevation']) as cursor:
    for smoothed_point in smoothed_points:
        x, y, z = smoothed_point
        cursor.insertRow([(x, y), z])

print("Kalman Filtering Completed")

Kalman Filtering Completed


In [3]:
#Create a new featureclass to hold the smoothed data
# Set the workspace (geodatabase or folder where the feature class will be created)
arcpy.env.workspace = r"C:\Users\purni\Documents\ArcGIS\Projects\For Phase 2_DEM_CS\Shape files"

# Set environment to overwrite existing outputs
arcpy.env.overwriteOutput = True

# Specify the feature class name
Output_FC = "WL_Kalmanfilter_XYZ_POINTS"

# Specify the geometry type (e.g., Point, Multipoint, Polygon, Polyline)
geometry_type = "POINT"

#specify spatial Reference
spatial_reference = arcpy.SpatialReference("WGS 1984")

#template = '#'  # Use an existing feature class as a template if needed

# Create the feature class
arcpy.CreateFeatureclass_management(arcpy.env.workspace, Output_FC, geometry_type, spatial_reference=spatial_reference)


# Specify the existing feature class name
#existing_feature_class = "ExistingFeatureClass"

# Specify the new field name and data type
new_field_name = "Elevation"
data_type = "FLOAT"  # You can choose other data types like "LONG", "DOUBLE", etc.

# Add the new field to the existing feature class
arcpy.AddField_management(Output_FC, new_field_name, data_type)

# Insert smoothed points into the output feature class
with arcpy.da.InsertCursor(Output_FC, ['SHAPE@XY', 'Elevation']) as cursor:
    for smoothed_point in smoothed_points:
        x, y, z = smoothed_point
        cursor.insertRow([(x, y), z])

print("Kalman Filtering Completed")

Kalman Filtering Completed
